# BrainAI on Colab (GPU)
Runtime > Change runtime type > T4 GPU. Everything runs on the session's local disk; download the checkpoint zip at the end.

In [1]:
# 1) repo + dependencies (no Google login needed)
!apt-get -qq install git-lfs && git lfs install
!git clone https://github.com/ZahidServers/BrainAI.git /content/BrainAI
%cd /content/BrainAI
!pip -q install torch datasets pillow opencv-python-headless transformers accelerate

Selecting previously unselected package git-lfs.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../git-lfs_3.4.1-1ubuntu0.4_amd64.deb ...
Unpacking git-lfs (3.4.1-1ubuntu0.4) ...
Setting up git-lfs (3.4.1-1ubuntu0.4) ...
Processing triggers for man-db (2.12.0-4build2) ...
Git LFS initialized.
Cloning into '/content/BrainAI'...
remote: Enumerating objects: 189, done.
remote: Counting objects: 100% (189/189), done.
remote: Compressing objects: 100% (146/146), done.
remote: Total 189 (delta 40), reused 178 (delta 35), pack-reused 0 (from 0)
Receiving objects: 100% (189/189), 40.19 MiB | 15.61 MiB/s, done.
Resolving deltas: 100% (40/40), done.
/content/BrainAI


In [2]:
# 2) storage: this session's local disk (no Google Drive needed)
import os
D = '/content/brainai'
os.makedirs(D + '/data', exist_ok=True)
print('storage:', D)

storage: /content/brainai


## 1. Text corpora (reading/writing)

In [ ]:
!python tools/fetch_text.py --out $D/data --mb 500
!ln -sf $D/data/*.txt data/   # corpus.py auto-discovers every data/*.txt as a domain

## 2. Sensor data (images, audio, video)

In [4]:
!python tools/fetch_sensors.py --out $D/sensors --n 20000
!ln -sfn $D/sensors sensors

README.md: 100% 3.90k/3.90k [00:00<00:00, 9.10MB/s]
dataset_infos.json: 100% 3.52k/3.52k [00:00<00:00, 8.69MB/s]

data/train-00000-of-00001-1359597a978bc4(…): downloading bytes:   1% 762k/146M [00:00<02:31, 962kB/s]
data/train-00000-of-00001-1359597a978bc4(…): downloading bytes:  93% 136M/146M [00:02<00:00, 88.3MB/s, 11.2MB/s  ]
data/train-00000-of-00001-1359597a978bc4(…): reconstructing file:  31% 45.7M/146M [00:02<00:04, 20.2MB/s,  145kB/s  ]
data/train-00000-of-00001-1359597a978bc4(…): downloading bytes: 100% 138M/138M [00:02<00:00, 50.4MB/s, 12.2MB/s  ]
data/train-00000-of-00001-1359597a978bc4(…): reconstructing file: 100% 146M/146M [00:02<00:00, 53.6MB/s, 13.7MB/s  ]

data/valid-00000-of-00001-70d52db3c749a9(…): downloading bytes:  68% 9.87M/14.6M [00:00<00:00, 18.7MB/s]
data/valid-00000-of-00001-70d52db3c749a9(…): downloading bytes: 100% 13.7M/13.7M [00:00<00:00, 21.2MB/s, 1.34MB/s  ]
data/valid-00000-of-00001-70d52db3c749a9(…): reconstructing file: 100% 14.6M/14.6M [00:00<00:00,

In [ ]:
!cp -r /content/brainai/* /content/BrainAI/
print("Done! Entire brainai folder has been copied and merged into BrainAI.")

## 2b. Robot data (state, actions, camera) - LeRobot format

In [6]:
!python tools/fetch_robot.py --out $D/robot_data --skip koch_pick_place_1_lego
!python tools/fetch_robot.py --out $D/robot_data --humanoid          # Unitree H1, ~1.4 GB
# optional, 9-18 GB each: !python tools/fetch_robot.py --out $D/robot_data --humanoid --humanoid-big
!ln -sfn $D/robot_data robot_data

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0% 0/8 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Reconstructing (incomplete total...):   0% 0.00/674k [00:00<?, ?B/s]          
Reconstructing (incomplete total...):   0% 0.00/783k [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/783k [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/7.67M [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/7.68M [00:00<?, ?B/s]

Fetching 8 files:  12% 1/8 [00:00<00:01,  5.34it/s]
Reconstructing (incomplete total...):   0% 3.54k/7.68M [00:00<06:42, 19.1kB/s]
Reconstructing (incomplete total...):   0% 5.80k/7.68M [00:00<06:42, 19.1kB/s]

Fetching 8 files:  38% 3/8 [00:00<00:00, 11.28it/s]
Reconstructing (incomplete total...):   0% 10.2k/7.69M [00:00<03:20, 38.4kB/s]

Fetching 8 files:  62% 5/8 [00:00<00:

## 2c. Train the perception encoders (held-out accuracy is printed vs chance)

In [1]:
# 1. Clear any corrupted sensor directory and ensure it is properly copied
!rm -rf /content/BrainAI/sensors
!mkdir -p /content/BrainAI/sensors
!cp -r /content/brainai/sensors/* /content/BrainAI/sensors/

# 2. Re-download if files are invalid, incomplete, or corrupted
import os
import zipfile
for f in ['images.npz', 'audio_sounds.npz', 'video.npz']:
    path = f"/content/BrainAI/sensors/{f}"
    if not os.path.exists(path) or os.path.getsize(path) < 1000:
        print(f"{f} is missing or corrupted. Refetching sensor data...")
        !python /content/BrainAI/tools/fetch_sensors.py --out /content/brainai/sensors --n 10000
        !cp -r /content/brainai/sensors/* /content/brainai/sensors/
        break
    try:
        with zipfile.ZipFile(path) as z:
            z.testzip()
    except Exception as e:
        print(f"{f} is invalid ({e}). Refetching...")
        !python /content/BrainAI/tools/fetch_sensors.py --out /content/brainai/sensors --n 10000
        !cp -r /content/brainai/sensors/* /content/brainai/sensors/
        break

# 3. Empty CUDA memory cache completely
import torch
torch.cuda.empty_cache()

# 4. Run the sensor training keeping the architecture scale at 'small'
# but reducing batch sizes for resource-heavy modalities (audio, video)
%cd /content/BrainAI
configs = [
    ('images', 500, 8, 'small'),
    ('audio', 500, 2, 'small'),
    ('video', 500, 8, 'small')
]

for m, s, batch, scale in configs:
    import torch
    torch.cuda.empty_cache()
    !PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python train_sensors.py --modality $m --steps $s --batch $batch --scale $scale

/content/BrainAI
images: 20000 samples, 200 classes, train 18000 / val 2000, device cuda
step     0  loss 5.606  val_acc 0.005  (8s)
step    50  loss 5.562  val_acc 0.009  (18s)
step   100  loss 5.320  val_acc 0.004  (29s)
step   150  loss 5.824  val_acc 0.007  (40s)
step   200  loss 5.146  val_acc 0.007  (50s)
step   250  loss 5.563  val_acc 0.004  (61s)
step   300  loss 5.422  val_acc 0.004  (72s)
step   350  loss 5.366  val_acc 0.004  (83s)
step   400  loss 5.246  val_acc 0.006  (93s)
step   450  loss 5.147  val_acc 0.009  (104s)
step   499  loss 5.318  val_acc 0.015  (115s)
FINAL images: val top-1 0.015   chance 0.005
audio: 2000 samples, 50 classes, train 1800 / val 200, device cuda
step     0  loss 3.954  val_acc 0.025  (20s)
step    50  loss 4.869  val_acc 0.015  (64s)
step   100  loss 4.988  val_acc 0.020  (109s)
step   150  loss 4.696  val_acc 0.015  (154s)
step   200  loss 4.523  val_acc 0.015  (199s)
step   250  loss 4.583  val_acc 0.040  (244s)
^C
video: 405 samples, 10 cla

## 2d. Behaviour cloning (state -> action); compares to copy-state baseline

In [2]:
# Execute robot behavior cloning pointing to the correct dataset directory
import torch
torch.cuda.empty_cache()
import glob

for d in sorted(glob.glob('/content/BrainAI/robot_data/*')):
    !PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python /content/BrainAI/train_robot.py --dataset $d --steps 1500 --batch 16 --scale small

aloha_sim_insertion_human: 50 episodes (45 train / 5 val), 22500 train frames, state 14-d, action 14-d, device cuda
step     0  train 1.8231  val 1.5974  (1s)
step   150  train 0.0392  val 0.0457  (8s)
step   300  train 0.0241  val 0.0318  (15s)
step   450  train 0.0117  val 0.0308  (22s)
step   600  train 0.0165  val 0.0234  (28s)
step   750  train 0.0136  val 0.0186  (35s)
step   900  train 0.0062  val 0.0183  (42s)
step  1050  train 0.0070  val 0.0153  (48s)
step  1200  train 0.0118  val 0.0178  (55s)
step  1350  train 0.0080  val 0.0159  (61s)
step  1499  train 0.0065  val 0.0163  (68s)
FINAL val MSE 0.0163 | mean-action baseline 1.0960 | copy-state baseline 0.2036
aloha_sim_transfer_cube_human: 50 episodes (45 train / 5 val), 18000 train frames, state 14-d, action 14-d, device cuda
step     0  train 1.6644  val 1.6337  (1s)
step   150  train 0.0332  val 0.0523  (8s)
step   300  train 0.0353  val 0.0390  (14s)
step   450  train 0.0139  val 0.0365  (20s)
step   600  train 0.0213  va

## 3. Distil Qwen into BrainAI (text; teacher writes, student trains)
Byte tokenizer != Qwen BPE, so logit-KL is impossible; we use sequence distillation.

In [ ]:
!python /content/BrainAI/distill.py generate --teacher Qwen/Qwen2.5-1.5B-Instruct --samples 50000

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 338/338 [00:11<00:00, 29.84it/s]
teacher: Qwen/Qwen2.5-1.5B-Instruct
generating 50,000 samples across 20 domains -> data/teacher_corpus.jsonl



## 4. Train (text + distilled). Start with `small`, move to `base` once loss curves look sane.

In [4]:
!python train_multidomain.py --scale small --steps 40000 --batch 32 --seq 256

BrainAI multi-domain training
BrainAI - built by Mohammed Zahid.

  [corpus] code_python ... 46,405 windows
  [corpus] aerodynamics_flight_dynamics_control_10000_examples ... Traceback (most recent call last):
  File "/content/BrainAI/train_multidomain.py", line 179, in <module>
    main()
    ~~~~^^
  File "/content/BrainAI/train_multidomain.py", line 69, in main
    corpus = load_multidomain(seq_len=args.seq, temperature=args.temperature,
                              seed=args.seed)
  File "/content/BrainAI/brainai/corpus.py", line 303, in load_multidomain
    g = torch.Generator().manual_seed(1234)
        ~~~~~~~~~~~~~~~^^
KeyboardInterrupt
^C


In [ ]:
# Local disk is wiped when the session ends: download the trained weights now.
!zip -qr brainai_checkpoints.zip checkpoints
from google.colab import files
files.download('brainai_checkpoints.zip')